# LitScope — Step 2: Classify Papers (SPECTER + DeepSeek)

**Pipeline:**
1. Load fetched papers from Google Drive
2. Generate SPECTER embeddings → saved for future semantic search
3. DeepSeek single-model classification
4. Results saved with confidence level

**Prerequisite:** Run `01_fetch_papers.ipynb` first

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Keep consistent with 01_fetch_papers.ipynb ────────────────────────────────
DRIVE_DIR = '/content/drive/MyDrive/LitScope/data'
# ─────────────────────────────────────────────────────────────────────────────

print(f'Data directory: {DRIVE_DIR}')

## 2. Install Dependencies

In [ ]:
!pip install requests pandas sentence-transformers -q

## 3. Configuration

In [ ]:
from google.colab import userdata

# ── API Keys ──────────────────────────────────────────────────────────────────
DEEPSEEK_API_KEY = userdata.get('DEEPSEEK_API_KEY')
# ─────────────────────────────────────────────────────────────────────────────

CLASSIFIED_CSV  = f'{DRIVE_DIR}/papers_classified.csv'
UNCERTAIN_CSV   = f'{DRIVE_DIR}/papers_uncertain.csv'
EMBEDDINGS_FILE = f'{DRIVE_DIR}/papers_embeddings.npy'
BY_PLATFORM_DIR = f'{DRIVE_DIR}/by_platform'

PLATFORM_ORDER = ['INFORMS / ISR', 'AIS / MISQ', 'Taylor & Francis / JMIS']

CSV_COLUMNS = [
    'title', 'abstract', 'year', 'venue', 'authors',
    'doi', 'url', 'source', 'platform',
    'is_behavioral', 'theories_used', 'confidence', 'reason',
]

print('Configuration complete')

## 4. Load Data and Check Status

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(CLASSIFIED_CSV)
df['year'] = pd.to_numeric(df['year'], errors='coerce')

# Papers pending classification: is_behavioral is NaN AND not already marked uncertain
pending_mask = df['is_behavioral'].isna() & (df['confidence'] != 'uncertain')

print(f'Total papers:          {len(df)}')
print(f'Already classified:    {(~df["is_behavioral"].isna()).sum()}')
print(f'Uncertain (skipped):   {(df["confidence"] == "uncertain").sum()}')
print(f'Pending classification: {pending_mask.sum()}')

## 5. Generate SciBERT Embeddings (SPECTER)

Generates semantic vectors for all papers using SPECTER — a SciBERT model fine-tuned on scientific paper pairs.
Embeddings are saved to Drive and will be used for semantic search in the future.
This step is skipped for papers that already have embeddings.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import os

# Load existing embeddings if available (to avoid re-encoding)
if os.path.exists(EMBEDDINGS_FILE):
    existing_embeddings = np.load(EMBEDDINGS_FILE)
    print(f'Loaded existing embeddings: {existing_embeddings.shape}')
    if existing_embeddings.shape[0] == len(df):
        embeddings = existing_embeddings
        print('All papers already have embeddings, skipping encoding.')
    else:
        print(f'Size mismatch ({existing_embeddings.shape[0]} vs {len(df)} papers), re-encoding all...')
        existing_embeddings = None
else:
    existing_embeddings = None
    print('No existing embeddings found, encoding all papers...')

if existing_embeddings is None or existing_embeddings.shape[0] != len(df):
    print('Loading SPECTER model...')
    specter = SentenceTransformer('allenai-specter')

    texts = [
        f"{str(row['title'])} [SEP] {str(row['abstract'])}"
        for _, row in df.iterrows()
    ]

    print(f'Encoding {len(texts)} papers (this may take a few minutes)...')
    embeddings = specter.encode(texts, batch_size=32, show_progress_bar=True)
    np.save(EMBEDDINGS_FILE, embeddings)
    print(f'Embeddings saved: {EMBEDDINGS_FILE}  shape={embeddings.shape}')

## 6. DeepSeek Classification

Single-model classification using DeepSeek (theory-first prompt).

Confidence levels:
- **high**: model confidence = high
- **medium**: model confidence = medium or low
- **uncertain**: API error / no valid response

> **Before running:** ensure `DEEPSEEK_API_KEY` is set in Colab Secrets.

In [ ]:
import requests
import json
import time

_DEEPSEEK_URL = 'https://api.deepseek.com/v1/chat/completions'

_PROMPT = """You are an expert reviewer of Information Systems (IS) academic papers.

Read only the TITLE and ABSTRACT. Determine whether this paper uses a behavioral or psychological theory as its PRIMARY theoretical foundation.

Title: {title}
Abstract: {abstract}

Behavioral IS papers typically show:
- Explicit theory names: TAM, TPB, UTAUT, Prospect Theory, SDT, Social Cognitive Theory, Trust Theory, Privacy Calculus, etc.
- Hypothesis language: "we hypothesize", "H1:", "proposed model", "theoretical model"
- Methods: survey, experiment, SEM, regression on human subjects

Non-behavioral IS papers typically show:
- Technical focus: algorithm, optimization, game theory, mechanism design, simulation
- No human behavioral angle: mathematical model, analytical model, pure ML/DL

Step 1 — Does the abstract name any behavioral or psychological theory?
Step 2 — Does it use hypothesis language or test a behavioral model?
Step 3 — Is the core contribution behavioral insight or technical/algorithmic?

Reply in JSON only:
{{"is_behavioral": true or false, "theories_used": ["only behavioral/psychological theories, empty if none"], "confidence": "high" or "medium" or "low", "reason": "one sentence citing evidence from the abstract"}}

Note: theories_used must only contain behavioral/psychological theories. Do NOT include game theory, information theory, or economic models."""


def classify_paper(title, abstract):
    prompt = _PROMPT.format(title=title, abstract=abstract)
    try:
        r = requests.post(
            _DEEPSEEK_URL,
            headers={'Authorization': f'Bearer {DEEPSEEK_API_KEY}', 'Content-Type': 'application/json'},
            json={
                'model':       'deepseek-chat',
                'messages':    [{'role': 'user', 'content': prompt}],
                'temperature': 0.1,
            },
            timeout=60
        )
        r.raise_for_status()
        content = r.json()['choices'][0]['message']['content']
        content = content.strip().replace('```json', '').replace('```', '').strip()
        return json.loads(content)
    except Exception as e:
        print(f'    [ERROR] {e}')
        return {'is_behavioral': None, 'theories_used': [], 'confidence': 'low', 'reason': f'Error: {e}'}

print('DeepSeek classifier defined')

In [ ]:
# ── Full classification loop ──────────────────────────────────────────────────
pending = df[pending_mask].index.tolist()
total   = len(pending)
print(f'Starting DeepSeek classification of {total} papers...\n')

for i, idx in enumerate(pending):
    title    = str(df.at[idx, 'title'])
    abstract = str(df.at[idx, 'abstract'])

    if len(abstract) < 50 or any(kw in title for kw in ('Special Section', 'Call for Papers', 'Editor')):
        df.at[idx, 'is_behavioral'] = False
        df.at[idx, 'theories_used'] = ''
        df.at[idx, 'confidence']    = 'high'
        df.at[idx, 'reason']        = 'Not a research paper'
        print(f'[{i+1}/{total}] Skipped: {title[:70]}')
        continue

    result = classify_paper(title, abstract)
    winner = result.get('is_behavioral')
    conf   = result.get('confidence', 'low')

    if winner is not None:
        df.at[idx, 'is_behavioral'] = winner
        df.at[idx, 'theories_used'] = ', '.join(result.get('theories_used', []))
        df.at[idx, 'confidence']    = conf
        df.at[idx, 'reason']        = result.get('reason', '')
        tag = '✅' if winner else '❌'
        print(f'[{i+1}/{total}] {tag} [{conf}] {title[:65]}')
        if result.get('theories_used'):
            print(f'    theories: {", ".join(result["theories_used"])}')
    else:
        df.at[idx, 'is_behavioral'] = None
        df.at[idx, 'theories_used'] = ''
        df.at[idx, 'confidence']    = 'uncertain'
        df.at[idx, 'reason']        = result.get('reason', '')
        print(f'[{i+1}/{total}] ⚠️  [uncertain] {title[:65]}')

    if (i + 1) % 20 == 0:
        df.to_csv(CLASSIFIED_CSV, index=False, encoding='utf-8-sig')
        print(f'  ── Auto-saved ({i+1}/{total}) ──')

    time.sleep(0.3)

print(f'\nClassification complete.')
print(f'  Classified:  {(~df["is_behavioral"].isna()).sum()} papers')
print(f'  Uncertain:   {(df["confidence"] == "uncertain").sum()} papers')

## 6b. Retry Uncertain Papers

Re-runs DeepSeek classification on all papers with `confidence == 'uncertain'`.
Useful when API quota ran out mid-batch — top up and run this cell without reprocessing everything.

In [ ]:
# ── Retry all uncertain papers ────────────────────────────────────────────────
uncertain_idx = df[df['confidence'] == 'uncertain'].index.tolist()
total = len(uncertain_idx)
print(f'Uncertain papers to retry: {total}\n')

for i, idx in enumerate(uncertain_idx):
    title    = str(df.at[idx, 'title'])
    abstract = str(df.at[idx, 'abstract'])

    if len(abstract) < 50 or any(kw in title for kw in ('Special Section', 'Call for Papers', 'Editor')):
        df.at[idx, 'is_behavioral'] = False
        df.at[idx, 'theories_used'] = ''
        df.at[idx, 'confidence']    = 'high'
        df.at[idx, 'reason']        = 'Not a research paper'
        print(f'[{i+1}/{total}] Skipped: {title[:70]}')
        continue

    result = classify_paper(title, abstract)
    winner = result.get('is_behavioral')
    conf   = result.get('confidence', 'low')

    if winner is not None:
        df.at[idx, 'is_behavioral'] = winner
        df.at[idx, 'theories_used'] = ', '.join(result.get('theories_used', []))
        df.at[idx, 'confidence']    = conf
        df.at[idx, 'reason']        = result.get('reason', '')
        tag = '✅' if winner else '❌'
        print(f'[{i+1}/{total}] {tag} [{conf}] {title[:65]}')
        if result.get('theories_used'):
            print(f'    theories: {", ".join(result["theories_used"])}')
    else:
        # Still failing — keep as uncertain
        df.at[idx, 'confidence'] = 'uncertain'
        df.at[idx, 'reason']     = result.get('reason', '')
        print(f'[{i+1}/{total}] ⚠️  [still uncertain] {title[:65]}')

    if (i + 1) % 20 == 0:
        df.to_csv(CLASSIFIED_CSV, index=False, encoding='utf-8-sig')
        print(f'  ── Auto-saved ({i+1}/{total}) ──')

    time.sleep(0.3)

# Final save
df.to_csv(CLASSIFIED_CSV, index=False, encoding='utf-8-sig')
still_uncertain = (df['confidence'] == 'uncertain').sum()
print(f'\nRetry complete.')
print(f'  Resolved:        {total - still_uncertain} / {total}')
print(f'  Still uncertain: {still_uncertain}')
print(f'Saved: {CLASSIFIED_CSV}')

## 7. Save Results and Split by Platform

In [ ]:
import os

# Sort by platform + year
order_map    = {p: i for i, p in enumerate(PLATFORM_ORDER)}
df['_order'] = df['platform'].map(order_map).fillna(len(PLATFORM_ORDER))
df.sort_values(['_order', 'year'], ascending=[True, False], inplace=True, na_position='last')
df.drop(columns=['_order'], inplace=True)
df.reset_index(drop=True, inplace=True)

# Save main classified CSV
df.to_csv(CLASSIFIED_CSV, index=False, encoding='utf-8-sig')
print(f'Saved: {CLASSIFIED_CSV}')

# Save uncertain papers separately for review
uncertain = df[df['confidence'] == 'uncertain'].copy()
if not uncertain.empty:
    uncertain.to_csv(UNCERTAIN_CSV, index=False, encoding='utf-8-sig')
    print(f'Uncertain papers saved: {UNCERTAIN_CSV}  ({len(uncertain)} papers)')

# Split by platform
os.makedirs(BY_PLATFORM_DIR, exist_ok=True)
for platform in PLATFORM_ORDER:
    sub = df[df['platform'] == platform].copy()
    if sub.empty:
        continue
    safe_name = platform.replace('/', '-').replace(' ', '_')
    sub.to_csv(f'{BY_PLATFORM_DIR}/{safe_name}.csv', index=False, encoding='utf-8-sig')

# Summary
print(f'\n{"="*55}')
print(f'  {"Platform":<26} {"Total":>6} {"Behavioral":>10} {"Uncertain":>9}')
print(f'  {"-"*53}')
for p in PLATFORM_ORDER:
    sub = df[df['platform'] == p]
    if sub.empty:
        continue
    b   = int((sub['is_behavioral'] == True).sum())
    u   = int((sub['confidence'] == 'uncertain').sum())
    yr  = (f"{int(sub['year'].min())}–{int(sub['year'].max())}"
           if sub['year'].notna().any() else 'N/A')
    print(f'  {p:<26} {len(sub):>6} {b:>10} {u:>9}   {yr}')
print(f'{"="*55}')
print(f'  {"Total":<26} {len(df):>6} {int((df["is_behavioral"]==True).sum()):>10} {len(uncertain):>9}')
print(f'\nAll done! Data saved to: {DRIVE_DIR}')

## 8. Full Validation — Keyword-Based Accuracy Check

Cross-validates all classified papers against a theory keyword list:
- **Recall**: among papers whose abstracts contain explicit theory keywords, % correctly predicted True
- **Precision**: among papers predicted True, % that contain keyword evidence
- **Missed papers**: lists False Negatives for manual review

In [ ]:
import re
import pandas as pd

# ── Reload classified CSV (in case this cell is run standalone) ───────────────
val_df = pd.read_csv(CLASSIFIED_CSV)
classified = val_df[val_df['is_behavioral'].notna() & (val_df['confidence'] != 'uncertain')].copy()

# ── Behavioral theory keywords — kept in sync with Prompt A/B ────────────────
BEHAVIORAL_KEYWORDS = [
    # TAM family
    r'technology acceptance model', r'\bTAM\b',
    # TPB / TRA
    r'theory of planned behavior', r'\bTPB\b',
    r'theory of reasoned action',  r'\bTRA\b',
    # UTAUT
    r'unified theory of acceptance', r'\bUTAUT2?\b',
    # Prospect Theory & bounded rationality family
    r'prospect theory',
    r'bounded rationality',
    r'framing effects?',
    r'anchoring and adjustment',
    r'mental accounting',
    # Motivation / self-regulation
    r'self.determination theory', r'\bSDT\b',
    r'regulatory focus theory',
    r'goal.setting theory',
    # Social theories
    r'social cognitive theory',
    r'social comparison theory',
    # IS-specific theories
    r'trust theory',
    r'privacy calculus',
    r'protection motivation theory',
    r'signaling theory',
    r'diffusion of innovation',
]

def has_keyword(text):
    text = str(text)
    return any(re.search(kw, text, re.IGNORECASE) for kw in BEHAVIORAL_KEYWORDS)

classified['has_keyword'] = classified['abstract'].apply(has_keyword)

# ── Ground truth subsets ──────────────────────────────────────────────────────
gt_true  = classified[classified['has_keyword'] == True]
gt_false = classified[classified['has_keyword'] == False]

# ── Recall: % of keyword-confirmed papers correctly predicted True ──────────
recall_correct = (gt_true['is_behavioral'] == True).sum()
recall_missed  = (gt_true['is_behavioral'] == False).sum()
recall = recall_correct / len(gt_true) * 100 if len(gt_true) > 0 else 0

# ── Precision: % of True predictions that have keyword evidence ─────────────
pred_true = classified[classified['is_behavioral'] == True]
precision_correct = pred_true['has_keyword'].sum()
precision = precision_correct / len(pred_true) * 100 if len(pred_true) > 0 else 0

# ── Confidence distribution ───────────────────────────────────────────────────
conf_counts = val_df['confidence'].value_counts()

# ── Print results ─────────────────────────────────────────────────────────────
print('=' * 60)
print('  Full Validation Results')
print('=' * 60)
print(f'\n  Total papers:               {len(val_df)}')
print(f'  Classified (non-uncertain): {len(classified)}')
print(f'  Uncertain:                  {(val_df["confidence"] == "uncertain").sum()}')

print(f'\n── Recall (keyword ground truth) ──────────────────────────')
print(f'  Papers with behavioral keywords: {len(gt_true)}')
print(f'  Correctly predicted True:        {recall_correct}  ({recall:.1f}%)')
print(f'  Missed (predicted False):        {recall_missed}  ({100-recall:.1f}%)')

print(f'\n── Precision (keyword check) ───────────────────────────────')
print(f'  Papers predicted True:           {len(pred_true)}')
print(f'  Confirmed by keyword:            {precision_correct}  ({precision:.1f}%)')
print(f'  No keyword (may still be valid): {len(pred_true)-precision_correct}  ({100-precision:.1f}%)')

print(f'\n── Confidence distribution ─────────────────────────────────')
for conf, count in conf_counts.items():
    print(f'  {conf:<12} {count:>5}  ({count/len(val_df)*100:.1f}%)')

print(f'\n── Missed behavioral papers (keyword=True but pred=False) ──')
missed_df = gt_true[gt_true['is_behavioral'] == False]
if missed_df.empty:
    print('  None ✅')
else:
    for _, row in missed_df.head(20).iterrows():
        print(f'  - {str(row["title"])[:75]}')
        print(f'    reason: {str(row["reason"])[:90]}')

print('\n' + '=' * 60)

## 9. Multi-Model Validation — Independent Precision Check

Randomly samples 50 papers from DeepSeek True predictions with no keyword match,
then validates them independently with GPT-OSS-120B (OpenRouter) and Llama-3.3-70B (Groq)
to estimate true precision without keyword bias.

**Prerequisites:** Set `OPENROUTER_API_KEY` and `GROQ_API_KEY` in Colab Secrets.

In [ ]:
import re, json, time, requests
import pandas as pd
from google.colab import userdata

OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
GROQ_API_KEY       = userdata.get('GROQ_API_KEY')

# ── OpenRouter model ID for GPT-OSS-120B ──────────────────────────────────────
# Verify the exact model slug at https://openrouter.ai/models
OPENROUTER_MODEL = 'openai/gpt-4o-mini'  # replace with GPT-OSS-120B slug if different

# ── Reload data and build sample ──────────────────────────────────────────────
val_df     = pd.read_csv(CLASSIFIED_CSV)
classified = val_df[
    val_df['is_behavioral'].notna() & (val_df['confidence'] != 'uncertain')
].copy()

BEHAVIORAL_KEYWORDS = [
    r'technology acceptance model', r'\bTAM\b',
    r'theory of planned behavior',  r'\bTPB\b',
    r'theory of reasoned action',   r'\bTRA\b',
    r'unified theory of acceptance', r'\bUTAUT2?\b',
    r'prospect theory', r'bounded rationality', r'framing effects?',
    r'anchoring and adjustment', r'mental accounting',
    r'self.determination theory', r'\bSDT\b',
    r'regulatory focus theory', r'goal.setting theory',
    r'social cognitive theory', r'social comparison theory',
    r'trust theory', r'privacy calculus', r'protection motivation theory',
    r'signaling theory', r'diffusion of innovation',
]

def has_keyword(text):
    return any(re.search(kw, str(text), re.IGNORECASE) for kw in BEHAVIORAL_KEYWORDS)

classified['has_keyword'] = classified['abstract'].apply(has_keyword)

# Target: papers predicted True by DeepSeek but with no keyword confirmation
no_kw_true = classified[
    (classified['is_behavioral'] == True) & (classified['has_keyword'] == False)
]
sample = no_kw_true.sample(n=min(50, len(no_kw_true)), random_state=42).reset_index(drop=True)
print(f'Population: {len(no_kw_true)} no-keyword True papers')
print(f'Sample:     {len(sample)} papers drawn (random_state=42)')

# ── Prompt B: method-first (independent of DeepSeek's Prompt A) ─────────────
_PROMPT_B = (
    "You are reviewing an Information Systems academic paper.\n\n"
    "Title: {title}\n"
    "Abstract: {abstract}\n\n"
    "Does this paper use a behavioral or psychological theory as its PRIMARY theoretical foundation?\n\n"
    "Behavioral IS paper signs:\n"
    "- Names a behavioral theory (TAM, TPB, UTAUT, SDT, Trust, Privacy Calculus, Social Cognitive Theory, etc.)\n"
    "- Tests hypotheses about human behavior, attitude, or intention\n"
    "- Uses survey or experiment on human subjects\n\n"
    "Non-behavioral paper signs:\n"
    "- Core is mathematical model, algorithm, or optimization\n"
    "- No human behavioral angle (game theory, econometrics only)\n\n"
    'Reply in JSON only, no other text:\n'
    '{{"is_behavioral": true or false, "reason": "one sentence citing evidence"}}'
)


def classify_openrouter(title, abstract):
    prompt = _PROMPT_B.format(title=title, abstract=abstract)
    try:
        r = requests.post(
            'https://openrouter.ai/api/v1/chat/completions',
            headers={
                'Authorization': f'Bearer {OPENROUTER_API_KEY}',
                'Content-Type':  'application/json',
            },
            json={
                'model':       OPENROUTER_MODEL,
                'messages':    [{'role': 'user', 'content': prompt}],
                'temperature': 0.1,
            },
            timeout=60,
        )
        r.raise_for_status()
        content = r.json()['choices'][0]['message']['content']
        content = content.strip().replace('```json', '').replace('```', '').strip()
        return json.loads(content)
    except Exception as e:
        return {'is_behavioral': None, 'reason': f'Error: {e}'}


def classify_groq(title, abstract):
    prompt = _PROMPT_B.format(title=title, abstract=abstract)
    try:
        r = requests.post(
            'https://api.groq.com/openai/v1/chat/completions',
            headers={
                'Authorization': f'Bearer {GROQ_API_KEY}',
                'Content-Type':  'application/json',
            },
            json={
                'model':       'llama-3.3-70b-versatile',
                'messages':    [{'role': 'user', 'content': prompt}],
                'temperature': 0.1,
            },
            timeout=60,
        )
        r.raise_for_status()
        content = r.json()['choices'][0]['message']['content']
        content = content.strip().replace('```json', '').replace('```', '').strip()
        return json.loads(content)
    except Exception as e:
        return {'is_behavioral': None, 'reason': f'Error: {e}'}


print('Classifiers ready: OpenRouter (GPT-OSS-120B) + Groq (Llama-3.3-70B)')
print(f'API keys present — OpenRouter: {bool(OPENROUTER_API_KEY)}, Groq: {bool(GROQ_API_KEY)}')


In [ ]:
# ── Run multi-model validation on 50 samples ──────────────────────────────────
results = []

for i, row in sample.iterrows():
    title    = str(row['title'])
    abstract = str(row['abstract'])

    gpt_res   = classify_openrouter(title, abstract)
    time.sleep(0.5)
    llama_res = classify_groq(title, abstract)
    time.sleep(0.3)

    ds_pred    = bool(row['is_behavioral'])
    gpt_pred   = gpt_res.get('is_behavioral')
    llama_pred = llama_res.get('is_behavioral')

    # Majority vote across 3 models (DeepSeek always True in this sample)
    votes    = [v for v in [ds_pred, gpt_pred, llama_pred] if v is not None]
    majority = (sum(votes) / len(votes) >= 0.5) if votes else None

    results.append({
        'title':        title,
        'deepseek':     ds_pred,
        'gpt':          gpt_pred,
        'llama':        llama_pred,
        'majority':     majority,
        'ds_reason':    str(row['reason']),
        'gpt_reason':   gpt_res.get('reason', ''),
        'llama_reason': llama_res.get('reason', ''),
    })

    g = '\u2705' if gpt_pred else ('\u274c' if gpt_pred is False else '?')
    l = '\u2705' if llama_pred else ('\u274c' if llama_pred is False else '?')
    m = '\u2705' if majority else '\u274c'
    print(f'[{i+1:>2}/50] DS=\u2705 GPT={g} Llama={l} \u2192 {m}  {title[:55]}')

results_df = pd.DataFrame(results)

# ── Summary report ─────────────────────────────────────────────────────────────
n          = len(results_df)
valid      = results_df.dropna(subset=['gpt', 'llama'])
both_true  = int(((valid['gpt'] == True)  & (valid['llama'] == True)).sum())
both_false = int(((valid['gpt'] == False) & (valid['llama'] == False)).sum())
split      = len(valid) - both_true - both_false
maj_true   = int(results_df['majority'].sum())
errors     = n - len(valid)

print(f'\n{"="*60}')
print(f'  Multi-Model Validation  (n={n})')
print(f'{"="*60}')
print(f'\n  All {n} papers: DeepSeek=True, no keyword match')
print(f'  API errors (excluded): {errors}')
print(f'\n\u2500\u2500 Validator agreement \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
print(f'  GPT + Llama both True:   {both_true:>3} / {len(valid)}  ({both_true/len(valid)*100:.1f}%)  \u2190 likely correct')
print(f'  GPT + Llama both False:  {both_false:>3} / {len(valid)}  ({both_false/len(valid)*100:.1f}%)  \u2190 likely false positives')
print(f'  Models disagree:         {split:>3} / {len(valid)}  ({split/len(valid)*100:.1f}%)  \u2190 borderline')
print(f'\n\u2500\u2500 Majority vote (3-model) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
print(f'  Confirmed True:  {maj_true:>3} / {n}  ({maj_true/n*100:.1f}%)')
print(f'  Overturned:      {n-maj_true:>3} / {n}  ({(n-maj_true)/n*100:.1f}%)')
print(f'\n\u2500\u2500 Interpretation \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
print(f'  Keyword-based precision was 8.3% (lower bound, keyword list incomplete)')
print(f'  Multi-model estimated precision: {maj_true/n*100:.1f}% (sample-based, n=50)')

# ── Papers overturned by validators ───────────────────────────────────────────
overturned = results_df[results_df['majority'] == False]
if not overturned.empty:
    print(f'\n\u2500\u2500 Likely False Positives (majority=False, n={len(overturned)}) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
    for _, r in overturned.iterrows():
        g = '\u2705' if r['gpt'] else ('\u274c' if r['gpt'] is False else '?')
        l = '\u2705' if r['llama'] else ('\u274c' if r['llama'] is False else '?')
        print(f'  [{g}/{l}] {r["title"][:70]}')
        print(f'       DeepSeek: {r["ds_reason"][:80]}')
        print(f'       GPT:      {r["gpt_reason"][:80]}')
        print()

print('=' * 60)
